# MODEL_CLUSTER — modèle Cluster pur

Notebook de lecture du pipeline canonique. Il ne lance **ni tuning ni export** : il charge le protocole, les paramètres, l’artefact existant et les résultats finaux.

Le modèle représente chaque utilisateur par ses moyennes de notes par genre, son âge et son sexe, standardise ces profils, puis applique KMeans. Une requête produit (âge, sexe, 1 à 3 films aimés) est affectée au centroïde le plus proche ; les films du groupe sont classés par overlap de genres puis score de groupe, avec un repli de popularité uniquement si le groupe fournit trop peu de candidats.


## 1. Protocole commun : 65 / 15 / 20

Le split déterministe (`random_state=42`) porte sur les interactions : 65 % entraînement, 15 % validation, 20 % test. Le tuning choisit les hyperparamètres sur `train → validation`. L’évaluation finale réentraîne sur `train+validation` et garde `test` intact.

La cible est **relative à l’utilisateur** : une interaction est aimée si `rating > user_mean(reference)`. La référence vaut `train` pendant la validation, puis `train+validation` pour le test final. Pour chaque cas évaluable, 1 à 3 films aimés de la référence sont tirés de façon déterministe.


In [1]:
import json
import sys
from pathlib import Path

ROOT = next(
    path for path in (Path.cwd(), Path.cwd().parent)
    if (path / "scripts" / "evaluation_protocol.py").exists()
)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from scripts.evaluation_protocol import build_eval_cases, load_dataframe, split_dataframe
from nextwatch_recommend import load, recommend

PARAMS_PATH = ROOT / "results" / "best_hyperparams_cluster_optuna.json"
RESULTS_PATH = ROOT / "results" / "final_evaluation.json"


In [2]:
data = load_dataframe()
split = split_dataframe(data)
validation_cases = build_eval_cases(split.train, split.validation)
test_cases = build_eval_cases(split.train_validation, split.test)

print(
    f"Interactions : train={len(split.train):,} | "
    f"validation={len(split.validation):,} | test={len(split.test):,}"
)
print(
    f"Cas produit : validation={len(validation_cases):,} | "
    f"test final={len(test_cases):,}"
)


Interactions : train=64,993 | validation=14,998 | test=19,999
Cas produit : validation=890 | test final=921


## 2. Paramètres sélectionnés

Le tuner Cluster pur a terminé **300 essais Optuna** sur la validation. Le meilleur essai est le numéro **240**, avec P@5 validation = **0,0467**. Le test final n’a pas participé à cette sélection.


In [3]:
params = json.loads(PARAMS_PATH.read_text(encoding="utf-8"))
parameter_names = [
    "MIN_SUPPORT", "N_CLUSTERS", "MIN_GROUP_CANDIDATES", "SEED_PROXY",
    "RATING_POWER", "SCORE_MODE", "GENRE_BOOST",
]

print(f"validation_score = {params['validation_score']:.6f}")
print("n_trials =", params["n_trials"])
print("best_trial_number =", params["best_trial_number"])
display({name: params[name] for name in parameter_names})


validation_score = 0.046742
n_trials = 300
best_trial_number = 240


{'MIN_SUPPORT': 45,
 'N_CLUSTERS': 17,
 'MIN_GROUP_CANDIDATES': 5,
 'SEED_PROXY': 5.0,
 'RATING_POWER': 0.5,
 'SCORE_MODE': 'sum',
 'GENRE_BOOST': 1.55}

## 3. Artefact d’inférence

L’export canonique entraîne le Cluster sur `train+validation`, sans utiliser le test. L’artefact stocke le scaler, les centroïdes KMeans, les scores film par cluster, le catalogue, les paramètres et les métadonnées de reproductibilité.


In [4]:
artifact = load(model="cluster")
meta = artifact["meta"]
metadata_fields = [
    "source", "model", "training_split", "test_used", "validation_score",
    "metric", "n_clusters", "n_users_profiles", "n_eligible", "n_titles",
    "dataset_sha256", "runtime_versions", "cluster_sizes",
]
display({name: meta.get(name) for name in metadata_fields})


{'source': 'results/best_hyperparams_cluster_optuna.json',
 'model': 'cluster_pure',
 'training_split': 'train+validation',
 'test_used': False,
 'validation_score': 0.04674157303370787,
 'metric': 'precision@5',
 'n_clusters': 17,
 'n_users_profiles': 943,
 'n_eligible': 569,
 'n_titles': 1680,
 'dataset_sha256': 'e1bd23ec38015edb2730a20c11666d38ffb729d848cd34cae5196144c6c0406e',
 'runtime_versions': {'python': '3.14.4',
  'numpy': '2.5.3',
  'pandas': '3.0.6',
  'scikit-learn': '1.9.1',
  'joblib': '1.6.0'},
 'cluster_sizes': {16: 128,
  1: 97,
  5: 86,
  8: 82,
  7: 79,
  0: 70,
  13: 62,
  2: 50,
  14: 49,
  4: 45,
  10: 44,
  11: 33,
  6: 29,
  12: 29,
  15: 25,
  3: 22,
  9: 13}}

## 4. Exemple d’inférence produit

L’API publique reçoit la démographie et un ou plusieurs films aimés. Le choix explicite `model="cluster"` garantit l’utilisation du modèle Cluster pur.


In [5]:
recommendations = recommend(
    age=24,
    gender="M",
    liked_movies=[50, 174],
    model="cluster",
    k=5,
)
recommendations


['Return of the Jedi (1983)',
 'Empire Strikes Back, The (1980)',
 'African Queen, The (1951)',
 'Starship Troopers (1997)',
 'Princess Bride, The (1987)']

## 5. Résultats finaux sur le test intact

Les trois méthodes sont évaluées sur les **mêmes 921 cas** avec l’inférence produit. La précision utilise un dénominateur fixe (`hits / k`). Le **Hit Rate@5** indique le pourcentage de cas avec au moins un film aimé dans le top 5. Ces scores proviennent exclusivement de `results/final_evaluation.json`.


In [6]:
final = json.loads(RESULTS_PATH.read_text(encoding="utf-8"))
assert final["n_cases"] == len(test_cases)

scores = final["scores"]
for method in ("baseline_overlap_mean", "user_cluster_groups", "user_user_cf"):
    values = scores[method]
    print(
        f"{method:24s} P@5={values['P@5']:.4f} | "
        f"P@10={values['P@10']:.4f} | Hit Rate@5={values['HitRate@5']:.2%}"
    )


baseline_overlap_mean    P@5=0.0343 | P@10=0.0372 | Hit Rate@5=15.53%
user_cluster_groups      P@5=0.0534 | P@10=0.0481 | Hit Rate@5=22.69%
user_user_cf             P@5=0.0932 | P@10=0.0853 | Hit Rate@5=37.57%


## Conclusion

Après 300 essais Optuna, le Cluster pur utilise **17 clusters** et atteint P@5 validation = **0,0467**. Sur le test final commun, il améliore la baseline : **P@5 0,0534 contre 0,0343** et **Hit Rate@5 22,7 % contre 15,5 %**. Il reste derrière UserCF, qui atteint **P@5 0,0932** et **Hit Rate@5 37,6 %**.

Le signal de groupe est utile, mais la compression des préférences en clusters perd encore de l’information par rapport au voisinage utilisateur fin.


In [7]:
# Cellule volontairement neutralisée.

<!-- Cellule volontairement neutralisée. -->


In [8]:
# Cellule volontairement neutralisée.


<!-- Cellule volontairement neutralisée. -->


In [9]:
# Cellule volontairement neutralisée.
